# Decentering vs sampling all coefficients

Two Discovery models on the same pulsar.

| | `inference=` | signals | sampler model | what is sampled |
|---|---|---|---|---|
| everyday | `"default"` (omit) | `discovery_signals()` | `decentered_model` | small nonlinear timing block + free noise hypers |
| full basis | `"all"` | `discovery_signals(joint=True)` | `joint_model` | every timing axis + GP coefficients |

A third mode (`numpyro.model` + `WhiteningConfig`) is in the package README.

AEI-DR2 combined J1022+1001. Expand at the prior center because this pulsar’s PX is negative. Short or no NUTS — the lesson is the dimension jump.


In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")

from pathlib import Path
import discovery as ds
from metapulsar import create_metapulsar
from nltiming import TimingSpec, TimingExpansionSpec
from nltiming.sampling import numpyro

numpyro.ensure_x64()

DATA = Path("..") / "data" / "J1022+1001"
pulsar = create_metapulsar(
    {"combined": [{
        "par": DATA / "J1022+1001.par",
        "tim": DATA / "J1022+1001.tim",
        "timing_package": "tempo2",
    }]},
    combination_strategy="per_pta",
    use_pulse_numbers="reuse",
)
nd = {f"{pulsar.name}_efac": 1.0, f"{pulsar.name}_log10_t2equad": -8.0}
print(pulsar.name, len(pulsar.toas))


## Default / decentered

`discovery_signals()` (not `joint=True`) keeps marginalized timing and GP coefficients inside the likelihood.


In [ ]:
spec_dec = TimingSpec(
    engines="jug", name="timing",
    expansion=TimingExpansionSpec.prior_center(),
)
timing_dec = spec_dec.for_pulsar(pulsar)
print(len(timing_dec.sampled), timing_dec.sampled)
print(len(timing_dec.marginalized), timing_dec.marginalized)

likelihood = ds.PulsarLikelihood([
    pulsar.residuals,
    ds.makenoise_measurement_simple(pulsar, nd),
    *timing_dec.discovery_signals(),
])
model_dec = numpyro.decentered_model(likelihood, timing_dec, fixed=nd)
print("xi dim:", model_dec.transport.dimension)


## Sample all / joint

`discovery_signals(joint=True)` requires a fully sampled plan — it raises if anything is marginalized.


In [ ]:
spec_all = TimingSpec(
    engines="jug", inference="all", name="timing",
    expansion=TimingExpansionSpec.prior_center(),
)
timing_all = spec_all.for_pulsar(pulsar)
print(len(timing_all.sampled), timing_all.sampled)

likelihood_all = ds.PulsarLikelihood([
    pulsar.residuals,
    ds.makenoise_measurement_simple(pulsar, nd),
    ds.makegp_fourier(pulsar, ds.powerlaw, 5, name="rednoise"),
    *timing_all.discovery_signals(joint=True),
])
fixed = {**nd, f"{pulsar.name}_rednoise_log10_A": -14.0, f"{pulsar.name}_rednoise_gamma": 3.0}
model_all = numpyro.joint_model(likelihood_all, timing_all, fixed=fixed)
print("xi dim:", model_all.transport.dimension, "hyper:", model_all.hyper_sites)
